In [1]:
%run ../shared_notebook_setup.py

## `tiktoken` Library Overview

`tiktoken` is OpenAI's fast tokenization library used to convert text into **tokens** (integer IDs) and back.  
It is useful for:

- Estimating prompt and response size before API calls
- Comparing tokenizer behavior across model encodings
- Managing context-window limits and cost

## Key Concepts

- **Token**: A chunk of text (not always a full word)
- **Encoding**: The tokenizer vocabulary/rules (e.g., `cl100k_base`, `o200k_base`)
- **Encode**: text → token IDs
- **Decode**: token IDs → text

## Common Workflow

1. Choose an encoding with `tiktoken.get_encoding(...)`
2. Encode text to token IDs
3. Inspect token count with `len(ids)`
4. Decode tokens (optionally per-token) to understand segmentation

## Why It Matters

Different encodings split text differently, which can change:

- Total token count
- Cost
- Effective context usage

In your notebook, comparing `cl100k_base` and `o200k_base` shows how the same sentence can produce different tokenizations and token counts.

## Example 1: Basic Tokenization Comparison

Let's compare how two different encodings tokenize the same text. Notice the differences in token count and segmentation patterns.

In [3]:
import tiktoken

# Compare token IDs and token text across encodings
sample_text = "Isn't this so unbelievably amazing?"

encodings = ["cl100k_base", "o200k_base"]
for encoding_name in encodings:
    encoding = tiktoken.get_encoding(encoding_name)
    ids = encoding.encode(sample_text)
    tokens = [encoding.decode([token_id]) for token_id in ids]

    print(f"\nEncoding: {encoding_name}")
    print(f"Text: {sample_text}")
    print(f"Token count: {len(ids)}")
    print("Token ID -> token text")
    for token_id, token_text in zip(ids, tokens):
        print(f"{token_id:>7} -> {repr(token_text)}")


Encoding: cl100k_base
Text: Isn't this so unbelievably amazing?
Token count: 8
Token ID -> token text
  89041 -> 'Isn'
    956 -> "'t"
    420 -> ' this'
    779 -> ' so'
  40037 -> ' unbelie'
  89234 -> 'vably'
   8056 -> ' amazing'
     30 -> '?'

Encoding: o200k_base
Text: Isn't this so unbelievably amazing?
Token count: 7
Token ID -> token text
   3031 -> 'Is'
   3023 -> "n't"
    495 -> ' this'
    813 -> ' so'
 180692 -> ' unbelievably'
   8467 -> ' amazing'
     30 -> '?'


## Example 2: Token Count Comparison Across Different Text Types

Different types of text tokenize differently. Let's see how code, numbers, and natural language compare:

In [4]:
import pandas as pd

# Different types of text to demonstrate tokenization patterns
test_texts = {
    "Short greeting": "Hi",
    "English sentence": "The quick brown fox jumps over the lazy dog.",
    "Python code": "def hello_world():\n    print('Hello, World!')",
    "JSON data": '{"name": "Alice", "age": 30, "active": true}',
    "URL": "https://github.com/openai/tiktoken/releases/tag/v0.5.0",
    "Numbers": "3.14159265359 × 1.23456789 = 3.87298334640",
    "Mixed content": "GPT-4 (model: gpt-4) costs $0.03/1K tokens 🚀",
    "Technical jargon": "Tokenization is the process of converting text into subword tokens.",
}

encoding = tiktoken.get_encoding("cl100k_base")
results = []

for text_type, text in test_texts.items():
    token_count = len(encoding.encode(text))
    char_count = len(text)
    results.append({
        "Type": text_type,
        "Text": text[:50] + "..." if len(text) > 50 else text,
        "Characters": char_count,
        "Tokens": token_count,
        "Ratio (chars/token)": round(char_count / token_count, 2) if token_count > 0 else 0
    })

df = pd.DataFrame(results)
print("\n=== Token Count Comparison ===\n")
print(df.to_string(index=False))
print("\nKey insight: Code and structured data typically require MORE tokens per character")
print("than natural language text.")


=== Token Count Comparison ===

            Type                                                  Text  Characters  Tokens  Ratio (chars/token)
  Short greeting                                                    Hi           2       1                 2.00
English sentence          The quick brown fox jumps over the lazy dog.          44      10                 4.40
     Python code        def hello_world():\n    print('Hello, World!')          45      12                 3.75
       JSON data          {"name": "Alice", "age": 30, "active": true}          44      17                 2.59
             URL https://github.com/openai/tiktoken/releases/tag/v0...          54      17                 3.18
         Numbers            3.14159265359 × 1.23456789 = 3.87298334640          42      21                 2.00
   Mixed content          GPT-4 (model: gpt-4) costs $0.03/1K tokens 🚀          44      24                 1.83
Technical jargon Tokenization is the process of converting text int... 

## Example 3: Understanding Special Tokens

Special tokens are reserved markers used by the model for structure (e.g., message boundaries, padding).

In [5]:
# Special tokens for chat completions
encoding = tiktoken.encoding_for_model("gpt-4")

# Special token IDs
special_tokens = {
    "<|im_start|>": encoding.encode_single_token("<|im_start|>") if "<|im_start|>" in encoding._special_tokens else None,
    "<|im_end|>": encoding.encode_single_token("<|im_end|>") if "<|im_end|>" in encoding._special_tokens else None,
}

print("=== Special Tokens in GPT-4 ===\n")

# Show all available special tokens
print(f"Total special tokens: {len(encoding._special_tokens)}")
print(f"\nSample special tokens:")
for token_str in list(encoding._special_tokens.keys())[:10]:
    try:
        token_id = encoding._special_tokens[token_str]
        print(f"  {repr(token_str):20} -> ID: {token_id}")
    except:
        pass

# Calculate token overhead for messages
print("\n=== Message Structure Token Count ===")
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is 2+2?"},
    {"role": "assistant", "content": "2+2 equals 4."},
]

total_tokens = 0
for msg in messages:
    role_tokens = len(encoding.encode(msg["role"]))
    content_tokens = len(encoding.encode(msg["content"]))
    msg_tokens = role_tokens + content_tokens + 4  # +4 for message delimiters
    total_tokens += msg_tokens
    print(f"{msg['role']:10} | content: {content_tokens:3} | total with overhead: {msg_tokens:3}")

print(f"\nTotal conversation tokens: {total_tokens}")

=== Special Tokens in GPT-4 ===

Total special tokens: 5

Sample special tokens:
  '<|endoftext|>'      -> ID: 100257
  '<|fim_prefix|>'     -> ID: 100258
  '<|fim_middle|>'     -> ID: 100259
  '<|fim_suffix|>'     -> ID: 100260
  '<|endofprompt|>'    -> ID: 100276

=== Message Structure Token Count ===
system     | content:   6 | total with overhead:  11
user       | content:   7 | total with overhead:  12
assistant  | content:   7 | total with overhead:  12

Total conversation tokens: 35


## Example 4: Model-to-Encoding Mapping

Different models use different encodings. Here's how to find the right encoding for your model:

In [6]:
# Different models use different encodings
models_and_encodings = [
    ("gpt-4", "cl100k_base"),
    ("gpt-4-turbo", "cl100k_base"),
    ("gpt-3.5-turbo", "cl100k_base"),
    ("text-davinci-003", "p50k_base"),
    ("text-davinci-002", "p50k_base"),
]

print("=== Model → Encoding Mapping ===\n")

sample_text = "Tokenization is fundamental to language models."

for model_name, expected_encoding in models_and_encodings:
    try:
        # Get encoding for this model
        encoding = tiktoken.encoding_for_model(model_name)
        actual_encoding = encoding.name
        token_count = len(encoding.encode(sample_text))
        
        match = "✓" if actual_encoding == expected_encoding else "✗"
        print(f"{match} {model_name:20} → {actual_encoding:15} ({token_count} tokens)")
    except KeyError:
        print(f"✗ {model_name:20} → Model not found in tiktoken")

print("\nTip: Use tiktoken.encoding_for_model(model_name) to automatically get")
print("the correct encoding for any OpenAI model.")

=== Model → Encoding Mapping ===

✓ gpt-4                → cl100k_base     (8 tokens)
✓ gpt-4-turbo          → cl100k_base     (8 tokens)
✓ gpt-3.5-turbo        → cl100k_base     (8 tokens)
✓ text-davinci-003     → p50k_base       (8 tokens)
✓ text-davinci-002     → p50k_base       (8 tokens)

Tip: Use tiktoken.encoding_for_model(model_name) to automatically get
the correct encoding for any OpenAI model.


## Example 5: Practical Cost Estimation

Estimate API costs before making requests by calculating token counts upfront:

In [7]:
# Estimate costs for API calls
encoding = tiktoken.encoding_for_model("gpt-4")

# Hypothetical pricing (check OpenAI docs for current rates)
GPT4_INPUT_COST = 0.03 / 1000    # $0.03 per 1K input tokens
GPT4_OUTPUT_COST = 0.06 / 1000   # $0.06 per 1K output tokens

# Sample prompts to estimate
prompts = [
    "What is machine learning?",
    "Explain the concept of tokenization in the context of NLP. Provide examples and explain why it matters for language models.",
    "Write a Python function that implements quicksort algorithm with detailed comments and examples.",
]

print("=== Cost Estimation for GPT-4 ===\n")
print(f"Input cost: ${GPT4_INPUT_COST*1000:.2f} per 1K tokens")
print(f"Output cost: ${GPT4_OUTPUT_COST*1000:.2f} per 1K tokens\n")

total_input_cost = 0

for i, prompt in enumerate(prompts, 1):
    input_tokens = len(encoding.encode(prompt))
    
    # Estimate output tokens (rough heuristic: assume output is 1.5x input)
    estimated_output = int(input_tokens * 1.5)
    
    input_cost = input_tokens * GPT4_INPUT_COST
    output_cost = estimated_output * GPT4_OUTPUT_COST
    total_cost = input_cost + output_cost
    
    total_input_cost += total_cost
    
    print(f"Prompt {i}:")
    print(f"  Input: {input_tokens:4} tokens → ${input_cost:.6f}")
    print(f"  Est. Output: {estimated_output:4} tokens → ${output_cost:.6f}")
    print(f"  Total: ${total_cost:.6f}\n")

print(f"Total estimated cost for all {len(prompts)} requests: ${total_input_cost:.6f}")
print("\nNote: This is a rough estimate. Actual output varies based on complexity.")

=== Cost Estimation for GPT-4 ===

Input cost: $0.03 per 1K tokens
Output cost: $0.06 per 1K tokens

Prompt 1:
  Input:    5 tokens → $0.000150
  Est. Output:    7 tokens → $0.000420
  Total: $0.000570

Prompt 2:
  Input:   25 tokens → $0.000750
  Est. Output:   37 tokens → $0.002220
  Total: $0.002970

Prompt 3:
  Input:   15 tokens → $0.000450
  Est. Output:   22 tokens → $0.001320
  Total: $0.001770

Total estimated cost for all 3 requests: $0.005310

Note: This is a rough estimate. Actual output varies based on complexity.


## Example 6: Edge Cases and Surprising Tokenization Patterns

Some texts tokenize in unexpected ways. Understanding these patterns helps optimize prompt engineering:

In [8]:
encoding = tiktoken.get_encoding("cl100k_base")

edge_cases = {
    "Whitespace matters": ["hello", " hello", "  hello", "hello "],
    "Punctuation impact": ["hello", "hello.", "hello!", "hello?"],
    "Case sensitivity": ["Hello", "hello", "HELLO"],
    "Contractions": ["dont", "don't", "do not"],
    "Numbers": ["123", "1,234", "1.234"],
    "Emoji": ["happy 😊", "sad 😢", "🚀"],
    "Unicode": ["café", "naïve", "résumé"],
}

print("=== Edge Cases in Tokenization ===\n")

for category, examples in edge_cases.items():
    print(f"{category}:")
    tokens_list = []
    for text in examples:
        tokens = encoding.encode(text)
        token_count = len(tokens)
        tokens_list.append((text, token_count, tokens))
        print(f"  '{text:15}' → {token_count} token(s): {tokens}")
    print()

print("\n=== Key Insights ===")
print("1. Leading/trailing whitespace affects tokenization")
print("2. Punctuation often requires separate tokens")
print("3. Contractions may tokenize differently than their expanded forms")
print("4. Special characters (emoji, unicode) can be expensive")
print("\nTip: Test your prompts with tiktoken to understand token usage!")

=== Edge Cases in Tokenization ===

Whitespace matters:
  'hello          ' → 1 token(s): [15339]
  ' hello         ' → 1 token(s): [24748]
  '  hello        ' → 2 token(s): [220, 24748]
  'hello          ' → 2 token(s): [15339, 220]

Punctuation impact:
  'hello          ' → 1 token(s): [15339]
  'hello.         ' → 2 token(s): [15339, 13]
  'hello!         ' → 2 token(s): [15339, 0]
  'hello?         ' → 2 token(s): [15339, 30]

Case sensitivity:
  'Hello          ' → 1 token(s): [9906]
  'hello          ' → 1 token(s): [15339]
  'HELLO          ' → 2 token(s): [51812, 1623]

Contractions:
  'dont           ' → 1 token(s): [78125]
  'don't          ' → 2 token(s): [15357, 956]
  'do not         ' → 2 token(s): [3055, 539]

Numbers:
  '123            ' → 1 token(s): [4513]
  '1,234          ' → 3 token(s): [16, 11, 11727]
  '1.234          ' → 3 token(s): [16, 13, 11727]

Emoji:
  'happy 😊        ' → 3 token(s): [57621, 27623, 232]
  'sad 😢          ' → 3 token(s): [83214, 27623, 95]


## Example 7: Encode/Decode Round-Trip

Verify that encoding and decoding work correctly, and understand information loss:

In [9]:
encoding = tiktoken.get_encoding("cl100k_base")

test_strings = [
    "Hello, World!",
    "Python   has   extra   spaces",
    "Line1\nLine2\nLine3",
    "price: $99.99",
    "email@example.com",
]

print("=== Encode/Decode Round-Trip Analysis ===\n")

for original in test_strings:
    # Encode to tokens
    token_ids = encoding.encode(original)
    
    # Decode back to string
    decoded = encoding.decode(token_ids)
    
    # Check if round-trip is exact
    exact_match = (original == decoded)
    
    print(f"Original:  {repr(original)}")
    print(f"Decoded:   {repr(decoded)}")
    print(f"Tokens:    {token_ids}")
    print(f"Match:     {exact_match}")
    
    if not exact_match:
        print(f"⚠️  Information may be lost in tokenization!")
    print()

print("Note: Most encoding/decoding is lossless, but edge cases exist.")
print("Always verify with your specific use case!")

=== Encode/Decode Round-Trip Analysis ===

Original:  'Hello, World!'
Decoded:   'Hello, World!'
Tokens:    [9906, 11, 4435, 0]
Match:     True

Original:  'Python   has   extra   spaces'
Decoded:   'Python   has   extra   spaces'
Tokens:    [31380, 256, 706, 256, 5066, 256, 12908]
Match:     True

Original:  'Line1\nLine2\nLine3'
Decoded:   'Line1\nLine2\nLine3'
Tokens:    [2519, 16, 198, 2519, 17, 198, 2519, 18]
Match:     True

Original:  'price: $99.99'
Decoded:   'price: $99.99'
Tokens:    [6692, 25, 400, 1484, 13, 1484]
Match:     True

Original:  'email@example.com'
Decoded:   'email@example.com'
Tokens:    [2386, 36587, 916]
Match:     True

Note: Most encoding/decoding is lossless, but edge cases exist.
Always verify with your specific use case!
